# AI in Go-To-Market (GTM) — Official-Source Evidence Report

This notebook scans **official company sources only** for verifiable, quote-level evidence
that AI is used in a company's **Go-To-Market (GTM) strategy**, over the **last 12 months**,
for an input list of companies.

## Workflow Overview

1. **Configure Environment**: Load API keys and initialize services
2. **Define Universe**: A list of company names (resolved to Bigdata entity IDs)
3. **Search Official Sources** (two passes per company, merged & deduplicated):
   - Pass A — document categories `filings` + `transcripts` (regulatory filings, annual /
     interim reports, earnings-call transcripts)
   - Pass B — source `DFF004` (PubT Corporate Communications: company-issued press releases)
   - General news media and sell-side research are **never** searched
4. **Extract Verbatim AI-GTM Quotes**: LLM extracts quote-level mentions where an explicit
   AI term ("AI", "artificial intelligence", "GenAI", "generative AI", "LLM",
   "machine learning") is directly linked to a GTM activity in the same/adjacent sentence
5. **Validate Deterministically**: every quote must (a) contain an explicit AI term
   (regex), (b) appear **verbatim** in the retrieved source text, (c) fall inside the
   12-month window; near-identical quotes within the same document are de-duplicated
6. **Strict Relevance Judge**: a second LLM pass audits each surviving quote against the
   AI-GTM rules and drops generic commentary, AI-as-end-market, and product-internal AI
   with no GTM link (when in doubt → drop)
7. **Count & Reconcile**: per-company counts are **computed from the citation rows**
   (never asked from the LLM), then reconciled
7. **Output**: two Markdown tables —
   **A) Company Summary** `| Company | Docs with AI-GTM Mentions | Total AI-GTM Mentions | Illustrative Quote | Citation |`
   **B) Citations** `| Company | Article/Document Title (verbatim) | Document Type | Date (ISO) | URL | Quote |`

## Exclusions (enforced in search filters + prompt + validation)

- Investor / capital-markets day decks (title filter + prompt exclusion)
- Third-party media/aggregators and sell-side notes (never searched)
- AI mentions about portfolio valuation, trading, or financial-market analysis
- Internal-only ops (finance/HR/IT) unless explicitly tied to GTM outcomes
- Generic AI claims; AI in customer products with no GTM link

## Step 1: Environment Setup

Load environment variables and verify the required API keys are present.

### Set up the Python environment with `uv`

```bash
# 1. Install uv (if not already installed)
curl -LsSf https://astral.sh/uv/install.sh | sh

# 2. From the notebook directory (AI_GTM_Evidence_Report/), create a virtual environment
uv venv

# 3. Activate it
source .venv/bin/activate        # macOS / Linux
# .venv\Scripts\activate         # Windows

# 4. Install dependencies
uv pip install -r requirements.txt

# 5. Launch Jupyter
uv run jupyter lab
```

> Requires Python 3.11+. Dependencies are listed in `requirements.txt`.

### Configure credentials

Either export `BIGDATA_API_KEY` / `OPENAI_API_KEY` in your shell, or create a `.env` file
(copy from `.env.example`):

```dotenv
# --- Required ---
BIGDATA_API_KEY=your_bigdata_api_key   # official-source search (Bigdata.com API)
OPENAI_API_KEY=your_openai_api_key     # verbatim quote extraction (LLM)

# --- Optional ---
#OPENAI_MODEL=gpt-4o-mini              # override the default OpenAI model
#LLM_PROVIDER=gemini                   # use Gemini instead of OpenAI
#GEMINI_API_KEY=your_gemini_api_key
```

In [1]:
import os
import sys
import json
import re
import asyncio
from pathlib import Path
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv

# Load environment variables (.env values do NOT override existing shell env vars)
load_dotenv()

openai_key = os.getenv('OPENAI_API_KEY')
bigdata_key = os.getenv('BIGDATA_API_KEY')

print(f"OpenAI API Key: {'Set' if openai_key else 'Missing'}")
print(f"Bigdata API Key: {'Set' if bigdata_key else 'Missing'}")

if not openai_key or not bigdata_key:
    print("\nPlease export the keys or create a .env file:")
    print("   OPENAI_API_KEY=your_key_here")
    print("   BIGDATA_API_KEY=your_key_here")

OpenAI API Key: Set
Bigdata API Key: Set


## Step 2: Initialize Services

Initialize the Bigdata search service and the LLM used for quote extraction, and load the
AI×GTM topic set plus the fixed vocabularies (AI terms, document types).

In [2]:
import importlib

# Ensure the notebook dir is importable and reload service modules so code
# changes take effect without a kernel restart.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import services.rate_limiter as _rate_limiter_module
import services.topic_search_service as _topic_search_module

importlib.reload(_rate_limiter_module)
importlib.reload(_topic_search_module)

from services.rate_limiter import AsyncSlidingWindowRateLimiter, AsyncConcurrencyLimiter
from services.topic_search_service import TopicSearchService
from services.llm_factory import LLMServiceFactory
from config.topics import (
    AI_GTM_TOPICS,
    OFFICIAL_DOC_CATEGORIES,
    PRESS_RELEASE_SOURCE_IDS,
    EXCLUDED_TITLE_PATTERNS,
    AI_TERMS,
    AI_TERM_REGEXES,
    DOCUMENT_TYPES,
)

# Search throughput settings (Search_Large_Scale profile): stay just under the
# 500 RPM API limit with a modest concurrency cap.
MAX_REQUESTS_PER_MINUTE = 480
SEARCH_WORKERS = 10

topic_search_service = TopicSearchService(
    api_key=bigdata_key,
    rate_limiter=AsyncSlidingWindowRateLimiter(max_requests=MAX_REQUESTS_PER_MINUTE),
    concurrency_limiter=AsyncConcurrencyLimiter(max_concurrent=SEARCH_WORKERS),
)

# LLM used for verbatim quote extraction (OpenAI by default, Gemini via LLM_PROVIDER)
llm_service = LLMServiceFactory.create(provider=os.getenv('LLM_PROVIDER', 'auto'))

print("Topic Search Service: Initialized")
print(f"LLM Service: Initialized (provider: {llm_service.provider_name}, model: {llm_service.model})")
print(f"\nAI x GTM topics loaded: {len(AI_GTM_TOPICS)}")
for i, topic in enumerate(AI_GTM_TOPICS, 1):
    print(f"  {i}. {topic['topic_name']}: {topic['topic_text'][:70]}...")
print(f"\nExplicit AI terms: {', '.join(AI_TERMS)}")
print(f"Document types: {', '.join(DOCUMENT_TYPES)}")

Topic Search Service: Initialized
LLM Service: Initialized (provider: openai, model: gpt-4o-mini)

AI x GTM topics loaded: 8
  1. Sales Execution & Enablement: {company} artificial intelligence AI sales force sales execution sales...
  2. Marketing & Personalization: {company} AI generative AI marketing campaigns digital marketing perso...
  3. Lead Gen & Customer Acquisition: {company} AI machine learning lead generation customer acquisition new...
  4. Pricing & Quote-to-Cash: {company} AI machine learning pricing optimization dynamic pricing quo...
  5. Channel, Partners & E-commerce: {company} AI artificial intelligence distributors channel partners dig...
  6. Product Launch & Commercialization: {company} launch AI-powered products artificial intelligence go-to-mar...
  7. Pipeline & Demand Forecasting: {company} AI machine learning sales pipeline demand forecasting demand...
  8. GTM Strategy (broad): {company} artificial intelligence generative AI go-to-market strategy ...

Expl

## Step 3: Configure Analysis Parameters

**The input is a list of company names** — edit `COMPANIES` below (or load it from a CSV).
The lookback window is fixed at 12 months per the research brief.

In [3]:
import pandas as pd

# ------------------------- INPUT: list of companies -------------------------
COMPANIES = ["Atlas Copco", "ABB", "Sandvik", "Schneider Electric"]

# Alternatively, load from a CSV with a COMPANY_NAME column:
# COMPANIES = pd.read_csv('companies.csv')['COMPANY_NAME'].dropna().tolist()
# -----------------------------------------------------------------------------

LOOKBACK_DAYS = 365          # last 12 months (per the research brief)
MAX_CHUNKS_PER_COMPANY = 120 # cap on retrieved excerpts sent to the LLM (by relevance)

end_date = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

print("Analysis Configuration:")
print(f"  Companies ({len(COMPANIES)}): {', '.join(COMPANIES)}")
print(f"  Period: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')} (last 12 months)")
print(f"  Pass A - official document categories: {OFFICIAL_DOC_CATEGORIES}")
print(f"  Pass B - press-release source filter (PubT): {PRESS_RELEASE_SOURCE_IDS}")
print(f"  Sentiment filter: None (GTM disclosures are frequently neutral)")
print(f"  Title exclusions: {EXCLUDED_TITLE_PATTERNS}")

Analysis Configuration:
  Companies (4): Atlas Copco, ABB, Sandvik, Schneider Electric
  Period: 2025-07-21 to 2026-07-21 (last 12 months)
  Pass A - official document categories: ['filings', 'transcripts']
  Pass B - press-release source filter (PubT): ['DFF004']
  Sentiment filter: None (GTM disclosures are frequently neutral)
  Title exclusions: ['capital markets day', 'capital market day', 'investor day', 'investor conference']


## Step 4: Resolve Companies to Bigdata Entities

Resolve each input name via the Knowledge Graph API and show the mapping so the universe
can be verified before searching.

> The raw API ranks **ticker** matches first, which is wrong for name inputs — e.g. the
> query "ABB" would resolve to *Aussie Broadband Ltd* (ASX ticker: ABB) instead of
> *ABB Ltd*. We therefore pick the best **name** match among the top candidates and seed
> the service cache so all subsequent searches reuse the corrected resolution.

In [4]:
import difflib as _difflib


async def resolve_company(name: str, top_n: int = 5):
    """Resolve a company name to the KG candidate whose NAME best matches the input."""
    data = await topic_search_service._api_post(
        "/knowledge-graph/companies",
        {"query": name, "types": ["PUBLIC"]},
        label=f"entity lookup {name}",
    )
    candidates = (data or {}).get("results", [])[:top_n]
    if not candidates:
        return None, []

    def name_score(c):
        cand = (c.get("name") or "").lower()
        query = name.lower()
        starts = 1 if cand.startswith(query) else 0
        return (starts, _difflib.SequenceMatcher(None, query, cand).ratio())

    best = max(candidates, key=name_score)
    # Seed the service cache so search_ticker() reuses this exact resolution.
    topic_search_service.company_cache.set(name, best["id"], best["name"])
    return best, candidates


rows = []
for name in COMPANIES:
    best, candidates = await resolve_company(name)
    others = ", ".join(c.get("name", "") for c in candidates
                       if not best or c.get("id") != best.get("id"))
    rows.append({
        "Input Name": name,
        "Resolved Name": best["name"] if best else "NOT FOUND",
        "Entity ID": best["id"] if best else "",
        "Other Candidates (rejected)": others[:120],
    })

df_entities = pd.DataFrame(rows)
display(df_entities)

unresolved = df_entities[df_entities["Entity ID"] == ""]["Input Name"].tolist()
if unresolved:
    print(f"WARNING - could not resolve: {unresolved} (they will be skipped)")
else:
    print("All companies resolved.")

# input name -> resolved name mapping used throughout the notebook
resolved_name_by_input = dict(zip(df_entities["Input Name"], df_entities["Resolved Name"]))

,Input Name,Resolved Name,Entity ID,Other Candidates (rejected)
0,Atlas Copco,Atlas Copco AB,9453BA,
1,ABB,ABB Ltd.,5FC63E,"Aussie Broadband Ltd., An Binh Commercial Join..."
2,Sandvik,Sandvik AB,554255,
3,Schneider Electric,Schneider Electric SE,990599,"Schneider National Inc., Schneider Electric In..."


All companies resolved.


## Step 5: Search Official Sources (two passes, merged)

For each company, run the AI×GTM topic set twice — **Pass A** restricted to the `filings` +
`transcripts` categories, **Pass B** restricted to source `DFF004` (official press
releases) — then merge and de-duplicate by document ID.

Two deterministic pre-filters are applied before extraction (dropped counts are reported):

- **Title exclusion** — investor / capital-markets day documents
- **AI-term pre-filter** — excerpts that contain no explicit AI term can never yield a
  valid quote, so they are dropped to keep the extraction context precise

In [5]:
import time


def passes_title_filter(headline: str) -> bool:
    """False when the document title matches an excluded pattern (investor day etc.)."""
    h = (headline or "").lower()
    return not any(pat in h for pat in EXCLUDED_TITLE_PATTERNS)


def contains_ai_term(text: str) -> bool:
    """True when the text contains at least one explicit AI term."""
    if not text:
        return False
    for pattern, case_sensitive in AI_TERM_REGEXES:
        flags = 0 if case_sensitive else re.IGNORECASE
        if re.search(pattern, text, flags):
            return True
    return False


async def search_company_official(company: str) -> dict:
    """Two-pass official-source search for one company (merged + pre-filtered)."""
    print(f"\nSearching official sources for {company}...")

    # Pass A: filings + transcripts (category filter, no source filter)
    pass_a = await topic_search_service.search_ticker(
        ticker=company,
        days=LOOKBACK_DAYS,
        custom_topics=AI_GTM_TOPICS,
        category_values=OFFICIAL_DOC_CATEGORIES,
        sentiment_values=None,
    )
    # Pass B: official press releases (PubT source filter DFF004)
    pass_b = await topic_search_service.search_ticker(
        ticker=company,
        days=LOOKBACK_DAYS,
        custom_topics=AI_GTM_TOPICS,
        source_ids=PRESS_RELEASE_SOURCE_IDS,
        sentiment_values=None,
    )

    for r in pass_a.get('topic_results', []):
        r['channel'] = r.get('document_type') or 'filings'
    for r in pass_b.get('topic_results', []):
        r['channel'] = 'press_release'

    merged = topic_search_service._deduplicate_across_topics(
        pass_a.get('topic_results', []) + pass_b.get('topic_results', [])
    )

    dropped_title, dropped_no_ai = 0, 0
    kept = []
    for r in merged:
        if not passes_title_filter(r.get('headline', '')):
            dropped_title += 1
            continue
        if not contains_ai_term(r.get('full_text') or r.get('summary', '')):
            dropped_no_ai += 1
            continue
        kept.append(r)

    kept = sorted(kept, key=lambda r: r.get('relevance', 0), reverse=True)
    dropped_cap = max(0, len(kept) - MAX_CHUNKS_PER_COMPANY)
    kept = kept[:MAX_CHUNKS_PER_COMPANY]

    company_name = pass_a.get('company_name') or pass_b.get('company_name') or company
    print(
        f"  {company}: pass A={len(pass_a.get('topic_results', []))}, "
        f"pass B={len(pass_b.get('topic_results', []))}, merged={len(merged)} -> kept={len(kept)} "
        f"(dropped: {dropped_title} excluded-title, {dropped_no_ai} no-AI-term, {dropped_cap} over-cap)"
    )
    return {
        "input_name": company,
        "company_name": company_name,
        "entity_id": pass_a.get('entity_id') or pass_b.get('entity_id'),
        "results": kept,
        "stats": {
            "pass_a_chunks": len(pass_a.get('topic_results', [])),
            "pass_b_chunks": len(pass_b.get('topic_results', [])),
            "merged": len(merged),
            "kept": len(kept),
            "dropped_excluded_title": dropped_title,
            "dropped_no_ai_term": dropped_no_ai,
            "dropped_over_cap": dropped_cap,
        },
    }


async def search_all_companies(companies, max_concurrent=2):
    semaphore = asyncio.Semaphore(max_concurrent)

    async def with_sem(c):
        async with semaphore:
            try:
                return await search_company_official(c)
            except Exception as e:
                print(f"  {c}: Error - {e}")
                return {"input_name": c, "company_name": c, "entity_id": None,
                        "results": [], "stats": {"error": str(e)}}

    results = await asyncio.gather(*[with_sem(c) for c in companies])
    return {r["input_name"]: r for r in results}


start_time = time.time()
search_results = await search_all_companies(COMPANIES, max_concurrent=2)
elapsed = time.time() - start_time

print(f"\n{'='*70}")
print(f"Search complete for {len(COMPANIES)} companies in {elapsed/60:.2f} minutes")
for name, data in search_results.items():
    print(f"  {data['company_name']}: {len(data['results'])} candidate excerpts")


Searching official sources for Atlas Copco...

Searching official sources for ABB...


  ABB: pass A=29, pass B=18, merged=47 -> kept=30 (dropped: 3 excluded-title, 14 no-AI-term, 0 over-cap)

Searching official sources for Sandvik...


  Atlas Copco: pass A=9, pass B=1, merged=10 -> kept=4 (dropped: 5 excluded-title, 1 no-AI-term, 0 over-cap)

Searching official sources for Schneider Electric...


  Schneider Electric: pass A=6, pass B=50, merged=56 -> kept=52 (dropped: 1 excluded-title, 3 no-AI-term, 0 over-cap)


  Sandvik: pass A=16, pass B=2, merged=18 -> kept=14 (dropped: 0 excluded-title, 4 no-AI-term, 0 over-cap)

Search complete for 4 companies in 0.26 minutes
  Atlas Copco AB: 4 candidate excerpts
  ABB Ltd.: 30 candidate excerpts
  Sandvik AB: 14 candidate excerpts
  Schneider Electric SE: 52 candidate excerpts


Persist the raw (pre-filtered) search results so the extraction step can be re-run
without re-querying the API.

In [6]:
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
search_results_file = output_dir / f'ai_gtm_search_results_{timestamp}.json'
with open(search_results_file, 'w') as f:
    json.dump(search_results, f, indent=2)
print(f"Saved search results: {search_results_file}")

Saved search results: output/ai_gtm_search_results_20260721_191916.json


## Step 6: Extract Verbatim AI-GTM Quotes (LLM)

The `ai_gtm_extract` prompt (see `config/prompts.yaml`) extracts **quote-level** rows:
a verbatim passage (≤ 2–3 sentences) where an explicit AI term is directly linked to a GTM
activity in the same or adjacent sentence, plus the document title (verbatim), document
type, ISO date, and URL. Multiple valid quotes per document are all captured; near-identical
quotes within a document are de-duplicated by the prompt (and again deterministically in
Step 7).

In [7]:
import yaml

with open('config/prompts.yaml', 'r', encoding='utf-8') as f:
    prompts = yaml.safe_load(f)
prompt_config = prompts['ai_gtm_extract']
SYSTEM_PROMPT = prompt_config['system_prompt']
USER_TEMPLATE = prompt_config['user_template']


def build_context(chunks: list) -> str:
    """Assemble the extraction context. The Date: line lets the model anchor the
    document date; Channel: helps it assign the Document Type."""
    parts = []
    for r in chunks:
        parts.append(f"Date: {(r.get('timestamp') or '')[:10]}")
        parts.append(f"Channel: {r.get('channel', '')}")
        parts.append(f"Source: {r.get('source', '')}")
        parts.append(f"Headline: {r.get('headline', '')}")
        parts.append(f"Content: {r.get('full_text') or r.get('summary', '')}")
        parts.append(f"Document URL: {r.get('document_url') or 'N/A'}")
        parts.append("")
    return "\n".join(parts)


async def extract_quotes_for_company(input_name: str, data: dict) -> dict:
    company_name = data.get('company_name', input_name)
    if not data.get('results'):
        return {"input_name": input_name, "company_name": company_name, "quotes": []}

    print(f"  Extracting AI-GTM quotes for {company_name} ({len(data['results'])} excerpts)...")
    context = build_context(data['results'])

    user_prompt = (
        USER_TEMPLATE
        .replace('{{current_datetime}}', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
        .replace('{{company_name}}', company_name)
        .replace('{{report}}', context)
        .replace('{{ai_terms}}', ", ".join(f'"{t}"' for t in AI_TERMS))
        .replace('{{document_types}}', " | ".join(DOCUMENT_TYPES))
    )
    full_prompt = f"{SYSTEM_PROMPT}\n\n{user_prompt}"

    try:
        response = await llm_service.generate_content_raw(prompt=full_prompt, model=None)
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', response)
        json_str = json_match.group(1).strip() if json_match else response.strip()
        quotes = json.loads(json_str)
        if isinstance(quotes, dict):
            quotes = quotes.get('quotes', quotes.get('mentions', []))
        quotes = [q for q in quotes if isinstance(q, dict) and q.get('is_relevant')]
        print(f"    {company_name}: {len(quotes)} candidate quotes")
        return {"input_name": input_name, "company_name": company_name, "quotes": quotes}
    except json.JSONDecodeError as e:
        print(f"    {company_name}: JSON parse error - {e}")
        return {"input_name": input_name, "company_name": company_name, "quotes": []}
    except Exception as e:
        print(f"    {company_name}: Error - {e}")
        return {"input_name": input_name, "company_name": company_name, "quotes": []}


async def extract_all(search_results: dict, max_concurrent: int = 4) -> list:
    semaphore = asyncio.Semaphore(max_concurrent)

    async def with_sem(input_name, data):
        async with semaphore:
            return await extract_quotes_for_company(input_name, data)

    print(f"Extracting AI-GTM quotes for {len(search_results)} companies...")
    print("=" * 60)
    return list(await asyncio.gather(
        *[with_sem(name, data) for name, data in search_results.items()]
    ))


start_time = time.time()
extraction_results = await extract_all(search_results, max_concurrent=4)
elapsed = time.time() - start_time

total_candidates = sum(len(r['quotes']) for r in extraction_results)
print(f"\n{'='*60}")
print(f"Extraction complete: {total_candidates} candidate quotes "
      f"across {len(extraction_results)} companies in {elapsed:.1f}s")

Extracting AI-GTM quotes for 4 companies...
  Extracting AI-GTM quotes for Atlas Copco AB (4 excerpts)...


  Extracting AI-GTM quotes for ABB Ltd. (30 excerpts)...
  Extracting AI-GTM quotes for Sandvik AB (14 excerpts)...
  Extracting AI-GTM quotes for Schneider Electric SE (52 excerpts)...


    Atlas Copco AB: 4 candidate quotes


    Sandvik AB: 6 candidate quotes


    ABB Ltd.: 4 candidate quotes


    Schneider Electric SE: 8 candidate quotes

Extraction complete: 22 candidate quotes across 4 companies in 14.6s


## Step 7: Deterministic Validation & De-duplication

Every candidate quote must survive four programmatic checks (LLM output is never trusted
blindly):

1. **Explicit AI term** — the quote must match one of the AI-term regexes
2. **Verbatim** — the (whitespace/quote-normalized) quote must appear in the retrieved
   source excerpts for that company (fuzzy fallback: longest common block ≥ 85% of quote)
3. **Window** — a parseable date must fall inside the last 12 months (future dates are
   invalid); unparseable dates are kept but blanked to `""`
4. **Near-duplicate de-dup** — within the same (company, document title), quotes with ≥ 85%
   similarity (or containment) collapse to the most complete phrasing

In [8]:
import difflib


def _normalize(s: str) -> str:
    """Normalize text for verbatim comparison: unify quotes/dashes, collapse whitespace, lowercase."""
    if not s:
        return ""
    s = (s.replace('‘', "'").replace('’', "'")
           .replace('“', '"').replace('”', '"')
           .replace('–', '-').replace('—', '-')
           .replace(' ', ' '))
    return re.sub(r'\s+', ' ', s).strip().lower()


def quote_is_verbatim(quote: str, chunk_norms: list) -> bool:
    """Exact normalized containment, else longest-common-block >= 85% of the quote."""
    qn = _normalize(quote)
    if len(qn) < 15:
        return False
    for cn in chunk_norms:
        if qn in cn:
            return True
    for cn in chunk_norms:
        m = difflib.SequenceMatcher(None, cn, qn, autojunk=False)
        block = m.find_longest_match(0, len(cn), 0, len(qn))
        if block.size >= 0.85 * len(qn):
            return True
    return False


def normalize_doc_type(dt: str, title: str = "") -> str:
    """Normalize to the fixed DOCUMENT_TYPES vocabulary. Title keywords take precedence
    over the LLM's assignment (deterministic beats generative)."""
    t = (title or "").lower()
    if "earnings call" in t or "transcript" in t:
        return "Earnings Call Transcript"
    if "sustainability report" in t or "esg report" in t:
        return "ESG / Sustainability Report"
    if "interim" in t or "half-year" in t or "half year" in t:
        return "Half-year / Interim Report"
    if "annual report" in t or "annual financial report" in t:
        return "Annual Report"
    dt = (dt or "").strip()
    if dt in DOCUMENT_TYPES:
        return dt
    low = dt.lower()
    for known in DOCUMENT_TYPES:
        if low == known.lower():
            return known
    return "Other Official Publication" if dt else ""


def validate_date(date_str: str) -> str:
    """Return ISO date if parseable; '' if blank/unparseable; 'OUT_OF_WINDOW' if outside 12m."""
    date_str = (date_str or "").strip()
    if not date_str:
        return ""
    try:
        d = datetime.strptime(date_str[:10], '%Y-%m-%d').replace(tzinfo=timezone.utc)
    except ValueError:
        return ""
    if d < start_date or d > end_date:
        return "OUT_OF_WINDOW"
    return d.strftime('%Y-%m-%d')


validated_rows = []
counts = {"candidates": 0, "dropped_no_ai_term": 0, "dropped_not_verbatim": 0,
          "dropped_out_of_window": 0, "dropped_near_dupes": 0}

for res in extraction_results:
    input_name = res['input_name']
    company_name = res['company_name']
    chunk_norms = [
        _normalize(r.get('full_text') or r.get('summary', ''))
        for r in search_results.get(input_name, {}).get('results', [])
    ]

    survivors = []
    for q in res['quotes']:
        counts["candidates"] += 1
        quote = (q.get('quote') or "").strip()

        # 1) explicit AI term (regex on the original quote)
        if not contains_ai_term(quote):
            counts["dropped_no_ai_term"] += 1
            continue
        # 2) verbatim in retrieved source text
        if not quote_is_verbatim(quote, chunk_norms):
            counts["dropped_not_verbatim"] += 1
            continue
        # 3) 12-month window
        date_iso = validate_date(q.get('date', ''))
        if date_iso == "OUT_OF_WINDOW":
            counts["dropped_out_of_window"] += 1
            continue

        survivors.append({
            "Company": company_name,
            "Article/Document Title (verbatim)": (q.get('document_title') or "").strip(),
            "Document Type": normalize_doc_type(q.get('document_type', ''),
                                                q.get('document_title', '')),
            "Date (ISO)": date_iso,
            "URL": (q.get('url') or "").strip(),
            "Quote": quote,
            "AI Term": q.get('ai_term', ''),
            "GTM Activity": q.get('gtm_activity', ''),
        })

    # 4) near-duplicate de-dup within the same document (keep most complete phrasing)
    survivors.sort(key=lambda r: len(r["Quote"]), reverse=True)
    kept = []
    for row in survivors:
        is_dupe = False
        for existing in kept:
            if existing["Article/Document Title (verbatim)"] != row["Article/Document Title (verbatim)"]:
                continue
            a, b = _normalize(row["Quote"]), _normalize(existing["Quote"])
            if a in b or b in a or difflib.SequenceMatcher(None, a, b).ratio() >= 0.85:
                is_dupe = True
                break
        if is_dupe:
            counts["dropped_near_dupes"] += 1
        else:
            kept.append(row)
    validated_rows.extend(kept)

print("Validation summary:")
for k, v in counts.items():
    print(f"  {k}: {v}")
print(f"  Rows surviving deterministic validation: {len(validated_rows)}")

Validation summary:
  candidates: 22
  dropped_no_ai_term: 3
  dropped_not_verbatim: 2
  dropped_out_of_window: 0
  dropped_near_dupes: 0
  Rows surviving deterministic validation: 17


## Step 7b: Strict Relevance Judge (second LLM pass)

Extraction is recall-oriented; this pass enforces **precision**. Each surviving quote is
audited (per company, in one batch call) against the strict AI-GTM rules — the AI term must
denote capability the company itself uses/embeds and be linked to *its own* GTM motion.
Generic commentary, thought-leadership, AI-as-end-market ("selling into AI data centers"),
and product-internal AI without a GTM link are dropped. When in doubt, the judge drops
(precision over recall). Dropped quotes are printed with the judge's reason for
auditability.

In [9]:
judge_config = prompts['ai_gtm_judge']
JUDGE_SYSTEM = judge_config['system_prompt']
JUDGE_TEMPLATE = judge_config['user_template']


async def judge_company_quotes(company_name: str, rows: list) -> list:
    """Audit a company's candidate quotes; return only the rows the judge keeps."""
    if not rows:
        return []
    candidates = [
        {"index": i,
         "document_title": r["Article/Document Title (verbatim)"],
         "document_type": r["Document Type"],
         "quote": r["Quote"]}
        for i, r in enumerate(rows)
    ]
    user_prompt = (
        JUDGE_TEMPLATE
        .replace('{{current_datetime}}', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
        .replace('{{company_name}}', company_name)
        .replace('{{quotes}}', json.dumps(candidates, indent=2, ensure_ascii=False))
        .replace('{{ai_terms}}', ", ".join(f'"{t}"' for t in AI_TERMS))
    )
    full_prompt = f"{JUDGE_SYSTEM}\n\n{user_prompt}"
    try:
        response = await llm_service.generate_content_raw(prompt=full_prompt, model=None)
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', response)
        json_str = json_match.group(1).strip() if json_match else response.strip()
        verdicts = json.loads(json_str)
        verdict_by_index = {v.get('index'): v for v in verdicts if isinstance(v, dict)}
    except Exception as e:
        print(f"  {company_name}: judge error - {e}; keeping rows unjudged")
        return rows

    kept = []
    for i, row in enumerate(rows):
        v = verdict_by_index.get(i)
        if v is None or v.get('keep'):
            kept.append(row)
        else:
            print(f"    DROP [{company_name}] \"{row['Quote'][:90]}...\" -> {v.get('reason', '')}")
    print(f"  {company_name}: judge kept {len(kept)}/{len(rows)}")
    return kept


rows_by_company = {}
for r in validated_rows:
    rows_by_company.setdefault(r["Company"], []).append(r)

print("Judging candidate quotes per company...")
judged_lists = await asyncio.gather(
    *[judge_company_quotes(company, rows) for company, rows in rows_by_company.items()]
)
judged_rows = [row for kept in judged_lists for row in kept]

print(f"\nJudge pass: {len(validated_rows)} -> {len(judged_rows)} quote rows")

CITATION_COLS = ["Company", "Article/Document Title (verbatim)", "Document Type",
                 "Date (ISO)", "URL", "Quote"]
df_citations = pd.DataFrame(judged_rows)
if not df_citations.empty:
    df_citations = df_citations.sort_values(
        ["Company", "Date (ISO)"], ascending=[True, False]
    ).reset_index(drop=True)
    display(df_citations[CITATION_COLS])
else:
    print("No validated AI-GTM quotes found for any company.")

Judging candidate quotes per company...


    DROP [ABB Ltd.] "ABB and SoftBank share the view that the world is entering a new era of AI-based robotics ..." -> AI as end-market demand, no GTM use
    DROP [ABB Ltd.] "The demand for AI in robotics is driven by the need for greater flexibility, faster commis..." -> Discussion of demand for AI in robotics, not linked to ABB's own GTM activities.
  ABB Ltd.: judge kept 1/3


    DROP [Schneider Electric SE] "this is where energy intelligence becomes essential. At Schneider Electric, we are advanci..." -> AI as end-market demand, no GTM use
    DROP [Schneider Electric SE] "AI enables true personalization at scale by processing vast amounts of customer data in re..." -> AI as end-market demand, no GTM use
    DROP [Schneider Electric SE] "enabling the construction of data centers from prefabricated units and addressing the comp..." -> AI usage not tied to specific GTM activity
    DROP [Schneider Electric SE] "the convergence of AI, IT/OT conversion integration, and digital platforms is reshaping wh..." -> Generic commentary on AI convergence, no specific GTM action
  Schneider Electric SE: judge kept 1/5
    DROP [Atlas Copco AB] "Several digital initiatives were also launched to strengthen the business area's digital p..." -> AI capability not linked to GTM activity
    DROP [Atlas Copco AB] "The latest in-house-developed hardware features an advanced con

,Company,Article/Document Title (verbatim),Document Type,Date (ISO),URL,Quote
0,ABB Ltd.,ABB Ltd - ABB adds Generative AI capabilities ...,Press Release,2026-03-30,https://new.abb.com/news/detail/134577/abb-add...,By integrating generative AI capabilities into...
1,Atlas Copco AB,Atlas Copco announces 2025 Annual Financial Re...,Annual Report,2026-03-20,https://files.captide.co/original-docs/c1f45ca...,Other digital initiatives included AI agents t...
2,Atlas Copco AB,Atlas Copco announces 2025 Annual Financial Re...,Annual Report,2026-03-20,https://files.captide.co/original-docs/c1f45ca...,"The newly launched ""S"" package brings AI-based..."
3,Sandvik AB,Sandvik Aktiebolag: Q4 2025 Earnings Call,Earnings Call Transcript,2026-01-27,,"In Intelligent Manufacturing, our Metrologic b..."
4,Sandvik AB,Sandvik AB (publ) announces 2025 Interim Finan...,Half-year / Interim Report,2026-01-27,https://files.captide.co/original-docs/f8fdd2a...,"Metrolog Copilot, an AI-based multilingual ass..."
5,Sandvik AB,Sandvik AB (publ) announces Q3 2025 Interim Fi...,Half-year / Interim Report,2025-10-20,https://files.captide.co/original-docs/f8fdd2a...,"Mastercam Copilot, launched in July, is an AI-..."
6,Schneider Electric SE,Schneider Electric SE - AI: How Companies in t...,Press Release,2025-09-12,https://blog.se.com/innovation/2025/09/12/ai-h...,"According to the study, AI has improved custom..."


## Step 8: Compute Counts & Reconcile

Counts are computed **from the citation rows** (never taken from the LLM):

- **Docs with AI-GTM Mentions** = distinct document titles per company
- **Total AI-GTM Mentions** = number of quote rows per company

Companies with no valid documents get a summary row with zeros and no citations.
Reconciliation then asserts that the summary counts exactly match the citations table.

In [10]:
summary_rows = []
for input_name in COMPANIES:
    company_name = resolved_name_by_input.get(input_name, input_name)
    comp = (df_citations[df_citations["Company"] == company_name]
            if not df_citations.empty else pd.DataFrame())

    if comp.empty:
        summary_rows.append({
            "Company": company_name,
            "Docs with AI-GTM Mentions": 0,
            "Total AI-GTM Mentions": 0,
            "Illustrative Quote": "",
            "Citation": "",
        })
        continue

    # Illustrative quote = the most complete (longest) validated quote for the company
    best = comp.loc[comp["Quote"].str.len().idxmax()]
    citation = (f"{best['Article/Document Title (verbatim)']} "
                f"({best['Document Type']}, {best['Date (ISO)']}, {best['URL']})")
    summary_rows.append({
        "Company": company_name,
        "Docs with AI-GTM Mentions": comp["Article/Document Title (verbatim)"].nunique(),
        "Total AI-GTM Mentions": len(comp),
        "Illustrative Quote": best["Quote"],
        "Citation": citation,
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

# ------------------------------ RECONCILIATION ------------------------------
print("\nReconciliation checks:")
all_ok = True
for _, row in df_summary.iterrows():
    comp = (df_citations[df_citations["Company"] == row["Company"]]
            if not df_citations.empty else pd.DataFrame())
    mentions_ok = row["Total AI-GTM Mentions"] == len(comp)
    docs_ok = row["Docs with AI-GTM Mentions"] == (
        comp["Article/Document Title (verbatim)"].nunique() if not comp.empty else 0
    )
    status = "PASS" if (mentions_ok and docs_ok) else "FAIL"
    all_ok &= (mentions_ok and docs_ok)
    print(f"  [{status}] {row['Company']}: "
          f"mentions {row['Total AI-GTM Mentions']} == {len(comp)} citation rows; "
          f"docs {row['Docs with AI-GTM Mentions']} == distinct titles")

# Every citation row must belong to a summary company (no orphans)
if not df_citations.empty:
    orphans = set(df_citations["Company"]) - set(df_summary["Company"])
    if orphans:
        all_ok = False
        print(f"  [FAIL] citation rows with no summary row: {orphans}")

assert all_ok, "Reconciliation failed - summary counts do not match citations table"
print("\nAll reconciliation checks passed.")

,Company,Docs with AI-GTM Mentions,Total AI-GTM Mentions,Illustrative Quote,Citation
0,Atlas Copco AB,1,2,Other digital initiatives included AI agents t...,Atlas Copco announces 2025 Annual Financial Re...
1,ABB Ltd.,1,1,By integrating generative AI capabilities into...,ABB Ltd - ABB adds Generative AI capabilities ...
2,Sandvik AB,3,3,"In Intelligent Manufacturing, our Metrologic b...",Sandvik Aktiebolag: Q4 2025 Earnings Call (Ear...
3,Schneider Electric SE,1,1,"According to the study, AI has improved custom...",Schneider Electric SE - AI: How Companies in t...



Reconciliation checks:
  [PASS] Atlas Copco AB: mentions 2 == 2 citation rows; docs 1 == distinct titles
  [PASS] ABB Ltd.: mentions 1 == 1 citation rows; docs 1 == distinct titles
  [PASS] Sandvik AB: mentions 3 == 3 citation rows; docs 3 == distinct titles
  [PASS] Schneider Electric SE: mentions 1 == 1 citation rows; docs 1 == distinct titles

All reconciliation checks passed.


## Step 9: Render Both Markdown Tables & Save Outputs

Renders **A) Company Summary Table** and **B) Citations Table** as Markdown, and saves
CSV/JSON/Markdown artifacts to `output/`.

In [11]:
from IPython.display import Markdown, display


def _md_escape(value) -> str:
    return str(value).replace('|', '\\|').replace('\n', ' ').strip()


def df_to_markdown_table(df: pd.DataFrame, cols: list) -> str:
    lines = ["| " + " | ".join(cols) + " |",
             "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(_md_escape(row.get(c, "")) for c in cols) + " |")
    return "\n".join(lines)


SUMMARY_COLS = ["Company", "Docs with AI-GTM Mentions", "Total AI-GTM Mentions",
                "Illustrative Quote", "Citation"]

report = f"""# AI in Go-To-Market (GTM) — Official-Source Evidence Report

**Period:** {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')} (last 12 months)
**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Sources:** official only — regulatory filings, annual/interim reports, ESG reports,
earnings-call transcripts, and company press releases (PubT `DFF004`). No third-party media,
no sell-side research, no investor/capital-markets day decks.

---

## A) Company Summary Table

{df_to_markdown_table(df_summary, SUMMARY_COLS)}

---

## B) Citations Table

{df_to_markdown_table(df_citations[CITATION_COLS] if not df_citations.empty else pd.DataFrame(columns=CITATION_COLS), CITATION_COLS)}

---

*Total AI-GTM mentions equal each company's citation rows; docs equal distinct titles
(reconciled programmatically in Step 8).*
"""

display(Markdown(report))

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

report_file = output_dir / f'ai_gtm_report_{timestamp}.md'
report_file.write_text(report, encoding='utf-8')
print(f"Report saved: {report_file}")

summary_csv = output_dir / f'ai_gtm_summary_{timestamp}.csv'
df_summary.to_csv(summary_csv, index=False)
print(f"Summary CSV saved: {summary_csv}")

if not df_citations.empty:
    citations_csv = output_dir / f'ai_gtm_citations_{timestamp}.csv'
    df_citations.to_csv(citations_csv, index=False)  # includes AI Term / GTM Activity audit columns
    print(f"Citations CSV saved: {citations_csv}")

quotes_json = output_dir / f'ai_gtm_quotes_{timestamp}.json'
with open(quotes_json, 'w') as f:
    json.dump(judged_rows, f, indent=2)
print(f"Validated quotes JSON saved: {quotes_json}")

# AI in Go-To-Market (GTM) — Official-Source Evidence Report

**Period:** 2025-07-21 to 2026-07-21 (last 12 months)
**Generated:** 2026-07-21 19:19:32
**Sources:** official only — regulatory filings, annual/interim reports, ESG reports,
earnings-call transcripts, and company press releases (PubT `DFF004`). No third-party media,
no sell-side research, no investor/capital-markets day decks.

---

## A) Company Summary Table

| Company | Docs with AI-GTM Mentions | Total AI-GTM Mentions | Illustrative Quote | Citation |
|---|---|---|---|---|
| Atlas Copco AB | 1 | 2 | Other digital initiatives included AI agents to further streamline software development, and the use of AI to provide technical support to customers within the service business. | Atlas Copco announces 2025 Annual Financial Report on Mar 20, 2026 (Annual Report, 2026-03-20, https://files.captide.co/original-docs/c1f45cae-e1c9-4e5f-b1a0-686262239922/c688e6a6-5e4b-4e13-96e4-d8abda580172.pdf?se=2300-05-06T16%3A10%3A40Z&sp=r&sv=2026-06-06&sr=b&sig=RBM7PzA/BiXbC29VMb9PCZoRd8xOTQGon%2BNVCAcrsb4%3D) |
| ABB Ltd. | 1 | 1 | By integrating generative AI capabilities into our Energy Management System, we are empowering teams to access the insights they need for faster, more informed decisions, without adding complexity to day-to-day operations. | ABB Ltd - ABB adds Generative AI capabilities to ABB Ability(TM) Energy Management System to accelerate operational insights (Press Release, 2026-03-30, https://new.abb.com/news/detail/134577/abb-adds-generative-ai-capabilities-to-abb-ability-energy-management-system-to-accelerate-operational-insights) |
| Sandvik AB | 3 | 3 | In Intelligent Manufacturing, our Metrologic business unit launched a new version of their software, where we now include our Copilot AI technology also in this software. | Sandvik Aktiebolag: Q4 2025 Earnings Call (Earnings Call Transcript, 2026-01-27, ) |
| Schneider Electric SE | 1 | 1 | According to the study, AI has improved customer acquisition success rates by up to 30% by identifying those actively seeking such solutions. | Schneider Electric SE - AI: How Companies in the Energy Sector Are Increasing Their Profits (Press Release, 2025-09-12, https://blog.se.com/innovation/2025/09/12/ai-how-companies-in-the-energy-sector-are-increasing-their-profits/) |

---

## B) Citations Table

| Company | Article/Document Title (verbatim) | Document Type | Date (ISO) | URL | Quote |
|---|---|---|---|---|---|
| ABB Ltd. | ABB Ltd - ABB adds Generative AI capabilities to ABB Ability(TM) Energy Management System to accelerate operational insights | Press Release | 2026-03-30 | https://new.abb.com/news/detail/134577/abb-adds-generative-ai-capabilities-to-abb-ability-energy-management-system-to-accelerate-operational-insights | By integrating generative AI capabilities into our Energy Management System, we are empowering teams to access the insights they need for faster, more informed decisions, without adding complexity to day-to-day operations. |
| Atlas Copco AB | Atlas Copco announces 2025 Annual Financial Report on Mar 20, 2026 | Annual Report | 2026-03-20 | https://files.captide.co/original-docs/c1f45cae-e1c9-4e5f-b1a0-686262239922/c688e6a6-5e4b-4e13-96e4-d8abda580172.pdf?se=2300-05-06T16%3A10%3A40Z&sp=r&sv=2026-06-06&sr=b&sig=RBM7PzA/BiXbC29VMb9PCZoRd8xOTQGon%2BNVCAcrsb4%3D | Other digital initiatives included AI agents to further streamline software development, and the use of AI to provide technical support to customers within the service business. |
| Atlas Copco AB | Atlas Copco announces 2025 Annual Financial Report on Mar 20, 2026 | Annual Report | 2026-03-20 | https://files.captide.co/original-docs/c1f45cae-e1c9-4e5f-b1a0-686262239922/c688e6a6-5e4b-4e13-96e4-d8abda580172.pdf?se=2300-05-06T16%3A10%3A40Z&sp=r&sv=2026-06-06&sr=b&sig=RBM7PzA/BiXbC29VMb9PCZoRd8xOTQGon%2BNVCAcrsb4%3D | The newly launched "S" package brings AI-based features that predict future air demand and machine behavior, further improving energy savings and pressure stability. |
| Sandvik AB | Sandvik Aktiebolag: Q4 2025 Earnings Call | Earnings Call Transcript | 2026-01-27 |  | In Intelligent Manufacturing, our Metrologic business unit launched a new version of their software, where we now include our Copilot AI technology also in this software. |
| Sandvik AB | Sandvik AB (publ) announces 2025 Interim Financial Report on Jan 27, 2026 | Half-year / Interim Report | 2026-01-27 | https://files.captide.co/original-docs/f8fdd2a3-8406-42cd-8f0a-8bdc9216f8ac/1be77a6e-6c34-4d43-8b80-b6ae33ea4d17.pdf?se=2299-12-08T02%3A03%3A11Z&sp=r&sv=2025-11-05&sr=b&sig=GuzKcGsAmFJMJhKbVsTtNDIMtNokEU3p1DXr1i1F1n4%3D | Metrolog Copilot, an AI-based multilingual assistant that guides process optimization. |
| Sandvik AB | Sandvik AB (publ) announces Q3 2025 Interim Financial Report on Oct 20, 2025 | Half-year / Interim Report | 2025-10-20 | https://files.captide.co/original-docs/f8fdd2a3-8406-42cd-8f0a-8bdc9216f8ac/1402dbcb-40df-4962-86be-406fc4faef29.pdf?se=2299-10-27T21%3A31%3A43Z&sp=r&sv=2025-11-05&sr=b&sig=H%2BXtg2OqzcAz24lkESWoVbAqrneIWa2n8jKexAhT9iE%3D | Mastercam Copilot, launched in July, is an AI-enabled assistant designed to provide contextual support and improve accessibility for users of all skill levels. |
| Schneider Electric SE | Schneider Electric SE - AI: How Companies in the Energy Sector Are Increasing Their Profits | Press Release | 2025-09-12 | https://blog.se.com/innovation/2025/09/12/ai-how-companies-in-the-energy-sector-are-increasing-their-profits/ | According to the study, AI has improved customer acquisition success rates by up to 30% by identifying those actively seeking such solutions. |

---

*Total AI-GTM mentions equal each company's citation rows; docs equal distinct titles
(reconciled programmatically in Step 8).*


Report saved: output/ai_gtm_report_20260721_191932.md
Summary CSV saved: output/ai_gtm_summary_20260721_191932.csv
Citations CSV saved: output/ai_gtm_citations_20260721_191932.csv
Validated quotes JSON saved: output/ai_gtm_quotes_20260721_191932.json


## Summary

The workflow is complete. Outputs generated in `output/`:

- `ai_gtm_report_*.md` — both Markdown tables (Company Summary + Citations)
- `ai_gtm_summary_*.csv` — per-company counts with illustrative quote
- `ai_gtm_citations_*.csv` — one row per validated verbatim quote (with AI-term /
  GTM-activity audit columns)
- `ai_gtm_quotes_*.json` — validated quote rows (full fidelity)
- `ai_gtm_search_results_*.json` — raw pre-filtered search excerpts (re-run extraction
  without re-querying the API)

**Guarantees:** counts are computed from citation rows and reconciled with assertions;
every quote contains an explicit AI term (regex-checked) and appears verbatim in the
retrieved official-source text; only the last 12 months and official sources are used.